In [ ]:
"""
DDIM Text-to-Image Diffusion Model for ImageNet
===============================================

A complete implementation of DDIM (Denoising Diffusion Implicit Models) for text-to-image generation
using CLIP text embeddings and ImageNet data. DDIM enables much faster sampling (10-50 steps vs 1000).

Features:
- DDIM sampling for fast generation
- CLIP text conditioning
- Classifier-free guidance
- Complete training and inference pipeline
- Interactive demo with step visualization
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_AVAILABLE = True
except ImportError:
    TENSORBOARD_AVAILABLE = False
    print("Warning: TensorBoard not available. Install with: pip install tensorboard")

import torchvision
from torchvision import transforms, datasets
from torchvision.utils import save_image, make_grid
from torchvision.models import inception_v3

import clip
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import warnings
import time
import json
import argparse
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Union, Any
from scipy import linalg
import math

# Suppress warnings
warnings.filterwarnings('ignore')

# =============================================================================
# HELPER FUNCTIONS AND MODULES
# =============================================================================

def space_to_depth(x, size=2):
    """Downscale method using depth dimension"""
    b, c, h, w = x.shape
    assert h % size == 0 and w % size == 0, "height/width must be divisible by size"
    out_h = h // size
    out_w = w // size
    out_c = c * (size * size)

    x = x.reshape((b, c, out_h, size, out_w, size))
    x = x.permute((0, 1, 3, 5, 2, 4))
    x = x.reshape((b, out_c, out_h, out_w))
    return x

class SpaceToDepth(nn.Module):
    def __init__(self, size, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.size = size

    def forward(self, x):
        return space_to_depth(x, self.size)

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, time_steps):
        positions = torch.unsqueeze(time_steps, 1)
        embeddings = torch.zeros((time_steps.shape[0], self.embedding_dim), device=time_steps.device)
        denominators = 10_000 ** (2 * torch.arange(self.embedding_dim // 2, device=time_steps.device) / self.embedding_dim)
        embeddings[:, 0::2] = torch.sin(positions / denominators)
        embeddings[:, 1::2] = torch.cos(positions / denominators)
        return embeddings

class WeightStandardizedConv2d(nn.Conv2d):
    """Weight Standardized Conv2d for improved training stability"""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True):
        super().__init__(in_channels, out_channels, kernel_size,
                         stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)

    def forward(self, x):
        eps = 1e-5 if x.dtype == torch.float32 else 1e-3
        weight = self.weight
        mean = weight.mean(dim=[1, 2, 3], keepdim=True)
        variance = weight.var(dim=[1, 2, 3], keepdim=True, correction=0)
        normalized_weight = (weight - mean) / torch.sqrt(variance + eps)
        return F.conv2d(x, normalized_weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

class Block(nn.Module):
    def __init__(self, in_channels, out_channels, groups=8):
        super().__init__()
        self.proj = WeightStandardizedConv2d(in_channels, out_channels, 3, padding=1)
        self.norm = nn.GroupNorm(groups, out_channels)
        self.act = nn.SiLU()

    def forward(self, x, scale_shift=None):
        x = self.proj(x)
        x = self.norm(x)
        if scale_shift is not None:
            scale, shift = scale_shift
            x = x * (scale + 1) + shift
        x = self.act(x)
        return x

class ResnetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_embed_dim=None, groups=8):
        super().__init__()
        if time_embed_dim is not None:
            self.mlp = nn.Sequential(
                nn.SiLU(),
                nn.Linear(time_embed_dim, 2 * out_channels)
            )
        else:
            self.mlp = None

        self.block1 = Block(in_channels, out_channels, groups)
        self.block2 = Block(out_channels, out_channels, groups)

        if in_channels == out_channels:
            self.res_conv = nn.Identity()
        else:
            self.res_conv = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x, time_embedding=None):
        scale_shift = None
        if self.mlp is not None and time_embedding is not None:
            time_emb = self.mlp(time_embedding)
            time_emb = time_emb.view(*time_emb.shape, 1, 1)
            scale_shift = time_emb.chunk(2, dim=1)

        h = self.block1(x, scale_shift=scale_shift)
        h = self.block2(h)
        return h + self.res_conv(x)

class Attention(nn.Module):
    def __init__(self, in_channels, num_heads=4, dim_head=32):
        super().__init__()
        self.num_heads = num_heads
        self.dim_head = dim_head
        self.scale_factor = 1 / (dim_head) ** 0.5
        self.hidden_dim = num_heads * dim_head
        self.input_to_qkv = nn.Conv2d(in_channels, 3 * self.hidden_dim, 1, bias=False)
        self.to_output = nn.Conv2d(self.hidden_dim, in_channels, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.input_to_qkv(x)
        q, k, v = map(lambda t: t.view(b, self.num_heads, self.dim_head, h * w), qkv.chunk(3, dim=1))
        q = q * self.scale_factor
        sim = torch.einsum("b h c i, b h c j -> b h i j", q, k)
        sim = sim - sim.amax(dim=-1, keepdim=True).detach()
        attention = sim.softmax(dim=-1)
        output = torch.einsum("b h i j, b h c j -> b h i c", attention, v)
        output = output.permute(0, 1, 3, 2).reshape((b, self.hidden_dim, h, w))
        return self.to_output(output)

class CrossAttention(nn.Module):
    """Cross attention between image features and text embeddings"""
    def __init__(self, query_dim: int, context_dim: int, num_heads: int = 8, dim_head: int = 64):
        super().__init__()
        self.num_heads = num_heads
        self.dim_head = dim_head
        self.scale = dim_head ** -0.5
        
        inner_dim = dim_head * num_heads
        
        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        self.to_k = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(context_dim, inner_dim, bias=False)
        
        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, query_dim),
            nn.Dropout(0.1)
        )
    
    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        
        # Reshape spatial dimensions to sequence
        x_seq = x.view(b, c, h * w).transpose(1, 2)  # (b, h*w, c)
        
        q = self.to_q(x_seq)
        k = self.to_k(context)
        v = self.to_v(context)
        
        # Reshape for multi-head attention
        q = q.view(b, h * w, self.num_heads, self.dim_head).transpose(1, 2)
        k = k.view(b, -1, self.num_heads, self.dim_head).transpose(1, 2)
        v = v.view(b, -1, self.num_heads, self.dim_head).transpose(1, 2)
        
        # Attention
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(b, h * w, -1)
        out = self.to_out(out)
        
        # Reshape back to spatial
        out = out.transpose(1, 2).view(b, c, h, w)
        
        return out + x  # Residual connection

# =============================================================================
# TEXT CONDITIONING MODULES
# =============================================================================

class TextEmbeddingProjector(nn.Module):
    """Projects CLIP text embeddings to model dimension"""
    def __init__(self, text_embed_dim: int = 512, model_embed_dim: int = 128):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(text_embed_dim, model_embed_dim * 2),
            nn.GELU(),
            nn.Linear(model_embed_dim * 2, model_embed_dim),
            nn.LayerNorm(model_embed_dim)
        )
    
    def forward(self, text_embeddings: torch.Tensor) -> torch.Tensor:
        return self.projection(text_embeddings)

class TextConditionalResnetBlock(ResnetBlock):
    """ResNet block with text conditioning via cross-attention"""
    def __init__(self, in_channels: int, out_channels: int, time_embed_dim: int,
                 text_embed_dim: int, groups: int = 8, use_cross_attn: bool = True):
        super().__init__(in_channels, out_channels, time_embed_dim, groups)
        
        self.use_cross_attn = use_cross_attn
        if use_cross_attn:
            self.cross_attn = CrossAttention(
                query_dim=out_channels,
                context_dim=text_embed_dim,
                num_heads=4,
                dim_head=out_channels // 4
            )
    
    def forward(self, x: torch.Tensor, time_embedding: Optional[torch.Tensor] = None,
                text_embedding: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Standard ResNet forward pass
        x = super().forward(x, time_embedding)
        
        # Apply cross-attention with text
        if self.use_cross_attn and text_embedding is not None:
            if text_embedding.dim() == 2:
                text_embedding = text_embedding.unsqueeze(1)  # Add sequence dimension
            x = self.cross_attn(x, text_embedding)
        
        return x

# =============================================================================
# U-NET ARCHITECTURE
# =============================================================================

class DownBlock(nn.Module):
    """Down block with text conditioning"""
    def __init__(self, in_ch: int, out_ch: int, time_emb_dim: int, text_emb_dim: int, use_attn: bool = False):
        super().__init__()
        self.block1 = TextConditionalResnetBlock(in_ch, out_ch, time_emb_dim, text_emb_dim, use_cross_attn=use_attn)
        self.block2 = TextConditionalResnetBlock(out_ch, out_ch, time_emb_dim, text_emb_dim, use_cross_attn=use_attn)
        self.attn = Attention(out_ch) if use_attn else nn.Identity()
        
        # Downsample using space-to-depth
        self.down = nn.Sequential(
            SpaceToDepth(2),
            nn.Conv2d(4 * out_ch, out_ch, 1)
        )

    def forward(self, x: torch.Tensor, time_cond: torch.Tensor, text_cond: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.block1(x, time_cond, text_cond)
        x = self.block2(x, time_cond, text_cond)
        x = self.attn(x)
        
        skip = x
        x = self.down(x)
        
        return x, skip

class UpBlock(nn.Module):
    """Up block with text conditioning"""
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, time_emb_dim: int, text_emb_dim: int, use_attn: bool = False):
        super().__init__()
        self.upconv = nn.ConvTranspose2d(in_ch, in_ch, kernel_size=4, stride=2, padding=1)
        self.block1 = TextConditionalResnetBlock(in_ch + skip_ch, out_ch, time_emb_dim, text_emb_dim, use_cross_attn=use_attn)
        self.block2 = TextConditionalResnetBlock(out_ch, out_ch, time_emb_dim, text_emb_dim, use_cross_attn=use_attn)
        self.attn = Attention(out_ch) if use_attn else nn.Identity()

    def forward(self, x: torch.Tensor, skip: torch.Tensor, time_cond: torch.Tensor, text_cond: torch.Tensor) -> torch.Tensor:
        x = self.upconv(x)
        x = torch.cat([x, skip], dim=1)
        x = self.block1(x, time_cond, text_cond)
        x = self.block2(x, time_cond, text_cond)
        x = self.attn(x)
        return x

class TextConditionalUNet(nn.Module):
    """U-Net with text conditioning for text-to-image diffusion"""
    
    def __init__(self, text_embed_dim: int = 512, model_channels: int = 128, 
                 resnet_depth: int = 4, image_size: int = 256, in_channels: int = 3):
        super().__init__()
        
        self.image_size = image_size
        self.in_channels = in_channels
        self.model_channels = model_channels
        
        # Time embedding
        time_emb_dim = model_channels
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 4),
            nn.GELU(),
            nn.Linear(time_emb_dim * 4, time_emb_dim)
        )
        
        # Text embedding projection
        self.text_projector = TextEmbeddingProjector(text_embed_dim, model_channels)
        
        # Initial convolution
        self.init_conv = nn.Conv2d(in_channels, model_channels, 3, padding=1)
        
        # Downsampling path
        self.downsample = nn.ModuleList()
        ch_list = []
        for i in range(resnet_depth):
            in_ch = model_channels if i == 0 else out_ch
            out_ch = model_channels * (2 ** i)
            ch_list.append(out_ch)
            
            self.downsample.append(
                DownBlock(
                    in_ch, out_ch, time_emb_dim, model_channels, 
                    use_attn=(i >= 2)  # Use attention in deeper layers
                )
            )
        
        # Bottleneck
        bottleneck_ch = ch_list[-1]
        self.bottleneck = nn.Sequential(
            TextConditionalResnetBlock(bottleneck_ch, bottleneck_ch, time_emb_dim, model_channels, use_cross_attn=True),
            TextConditionalResnetBlock(bottleneck_ch, bottleneck_ch, time_emb_dim, model_channels, use_cross_attn=True),
            Attention(bottleneck_ch)
        )
        
        # Upsampling path
        self.upsample = nn.ModuleList()
        for i in range(resnet_depth - 1, -1, -1):
            in_ch = ch_list[i] if i == resnet_depth - 1 else out_ch
            skip_ch = ch_list[i]
            out_ch = skip_ch if i > 0 else model_channels
            
            self.upsample.append(
                UpBlock(
                    in_ch, skip_ch, out_ch, time_emb_dim, model_channels,
                    use_attn=(i >= 2)
                )
            )
        
        # Final output layers
        self.final_block = nn.Sequential(
            TextConditionalResnetBlock(model_channels, model_channels, time_emb_dim, model_channels, use_cross_attn=True),
            nn.GroupNorm(8, model_channels),
            nn.SiLU(),
            nn.Conv2d(model_channels, in_channels, 3, padding=1)
        )
    
    def forward(self, x: torch.Tensor, timesteps: torch.Tensor, 
                text_embeddings: torch.Tensor, cfg_scale: float = 1.0) -> torch.Tensor:
        
        # Handle classifier-free guidance during training
        if cfg_scale != 1.0 and self.training:
            # During training, randomly replace some text embeddings with zeros for CFG
            batch_size = text_embeddings.size(0)
            cfg_mask = torch.rand(batch_size, device=text_embeddings.device) < 0.1  # 10% dropout
            text_embeddings = text_embeddings.clone()
            text_embeddings[cfg_mask] = 0
        
        # Time and text embeddings
        time_emb = self.time_mlp(timesteps)
        text_emb = self.text_projector(text_embeddings)
        
        # Initial convolution
        x = self.init_conv(x)
        
        # Downsampling
        skips = []
        for layer in self.downsample:
            x, skip = layer(x, time_emb, text_emb)
            skips.append(skip)
        
        # Bottleneck
        for i, layer in enumerate(self.bottleneck):
            if isinstance(layer, TextConditionalResnetBlock):
                x = layer(x, time_emb, text_emb)
            else:
                x = layer(x)
        
        # Upsampling
        for i, layer in enumerate(self.upsample):
            skip = skips[-(i+1)]
            x = layer(x, skip, time_emb, text_emb)
        
        # Final output
        if isinstance(self.final_block[0], TextConditionalResnetBlock):
            x = self.final_block[0](x, time_emb, text_emb)
            x = self.final_block[1:](x)
        else:
            x = self.final_block(x)
        
        return x

# =============================================================================
# DDIM SCHEDULER
# =============================================================================

class DDIMScheduler:
    """DDIM (Denoising Diffusion Implicit Models) Scheduler for fast sampling"""
    def to(self, device):
        """Move scheduler tensors to specified device"""
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device) 
        self.alphas_cumprod = self.alphas_cumprod.to(device)
        self.alphas_cumprod_prev = self.alphas_cumprod_prev.to(device)
        if hasattr(self, 'timesteps'):
            self.timesteps = self.timesteps.to(device)
        return self
    
    def __init__(self, num_train_timesteps: int = 1000, beta_start: float = 0.0001, 
                 beta_end: float = 0.02, beta_schedule: str = "linear"):
        self.num_train_timesteps = num_train_timesteps
        
        # Create beta schedule
        if beta_schedule == "linear":
            self.betas = torch.linspace(beta_start, beta_end, num_train_timesteps, dtype=torch.float32)
        elif beta_schedule == "cosine":
            self.betas = self._cosine_schedule(num_train_timesteps)
        else:
            raise ValueError(f"Unknown beta_schedule: {beta_schedule}")
        
        # Precompute useful quantities
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), self.alphas_cumprod[:-1]])
        
        # For DDIM sampling
        self.final_alpha_cumprod = 1.0
        
    def _cosine_schedule(self, timesteps, s=0.008):
        """Cosine noise schedule"""
        def f(t):
            return torch.cos((t / timesteps + s) / (1 + s) * 0.5 * torch.pi) ** 2
        x = torch.linspace(0, timesteps, timesteps + 1)
        alphas_cumprod = f(x) / f(torch.tensor([0]))
        betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
        return torch.clip(betas, 0.0001, 0.9999)
    
    def set_timesteps(self, num_inference_steps: int, device: torch.device = None):
        """Set the timesteps for DDIM sampling"""
        self.num_inference_steps = num_inference_steps
        
        # Create inference schedule - evenly spaced timesteps
        step_ratio = self.num_train_timesteps // num_inference_steps
        timesteps = (torch.arange(0, num_inference_steps) * step_ratio).round().long()
        timesteps = torch.flip(timesteps, dims=[0])  # Reverse for denoising
        
        self.timesteps = timesteps
        if device is not None:
            self.timesteps = self.timesteps.to(device)
            self.alphas_cumprod = self.alphas_cumprod.to(device)
    
    def add_noise(self, original_samples: torch.Tensor, noise: torch.Tensor, 
                timesteps: torch.Tensor) -> torch.Tensor:
        """Add noise to samples (forward process)"""
        # FIX: Ensure alphas_cumprod is on the same device as timesteps
        alphas_cumprod = self.alphas_cumprod.to(timesteps.device)
        
        sqrt_alpha_prod = alphas_cumprod[timesteps] ** 0.5
        sqrt_one_minus_alpha_prod = (1 - alphas_cumprod[timesteps]) ** 0.5
        
        # Reshape for broadcasting
        sqrt_alpha_prod = sqrt_alpha_prod.flatten()
        while len(sqrt_alpha_prod.shape) < len(original_samples.shape):
            sqrt_alpha_prod = sqrt_alpha_prod.unsqueeze(-1)
        
        sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.flatten()
        while len(sqrt_one_minus_alpha_prod.shape) < len(original_samples.shape):
            sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.unsqueeze(-1)
        
        noisy_samples = sqrt_alpha_prod * original_samples + sqrt_one_minus_alpha_prod * noise
        return noisy_samples
    
    def step(self, model_output: torch.Tensor, timestep: int, sample: torch.Tensor,
             eta: float = 0.0, use_clipped_model_output: bool = False) -> torch.Tensor:
        """
        Perform one DDIM denoising step
        
        Args:
            model_output: Direct output from the learned diffusion model
            timestep: Current discrete timestep in the diffusion chain
            sample: Current instance of sample being created by diffusion process
            eta: Weight of noise for DDIM sampling (0.0 = deterministic, 1.0 = DDPM)
        """
        
        # 1. get previous step value (=t-1)
        prev_timestep = timestep - self.num_train_timesteps // self.num_inference_steps
        
        # 2. compute alphas, betas
        alpha_prod_t = self.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else self.final_alpha_cumprod
        
        beta_prod_t = 1 - alpha_prod_t
        
        # 3. compute predicted original sample from predicted noise
        if use_clipped_model_output:
            model_output = torch.clamp(model_output, -1, 1)
        
        pred_original_sample = (sample - beta_prod_t ** (0.5) * model_output) / alpha_prod_t ** (0.5)
        
        # 4. Clip predicted x_0
        pred_original_sample = torch.clamp(pred_original_sample, -1, 1)
        
        # 5. compute variance: "sigma_t(η)" -> see formula (16)
        variance = self._get_variance(timestep, prev_timestep, eta)
        std_dev_t = variance ** (0.5)
        
        # 6. compute "direction pointing to x_t" of formula (12)
        pred_sample_direction = (1 - alpha_prod_t_prev - std_dev_t**2) ** (0.5) * model_output
        
        # 7. compute x_t without "random noise" of formula (12)
        prev_sample = alpha_prod_t_prev ** (0.5) * pred_original_sample + pred_sample_direction
        
        if eta > 0:
            noise = torch.randn_like(sample)
            prev_sample = prev_sample + std_dev_t * noise
        
        return prev_sample
    
    def _get_variance(self, timestep, prev_timestep, eta):
        """Compute variance for DDIM sampling"""
        alpha_prod_t = self.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else self.final_alpha_cumprod
        beta_prod_t = 1 - alpha_prod_t
        beta_prod_t_prev = 1 - alpha_prod_t_prev
        
        variance = (beta_prod_t_prev / beta_prod_t) * (1 - alpha_prod_t / alpha_prod_t_prev)
        variance = variance * eta ** 2
        
        return variance

# =============================================================================
# DATA LOADING
# =============================================================================

class ImageNetMiniDataset(Dataset):
    """ImageNet Mini dataset with CLIP text embeddings"""
    
    def __init__(self, root_dir: str, clip_model, transform=None, split='train', 
                 max_classes: Optional[int] = None):
        self.root_dir = Path(root_dir)
        self.clip_model = clip_model
        self.transform = transform
        self.split = split
        
        # Load ImageNet class mappings from words.txt
        self.class_to_name = self._load_class_mappings()
        
        # Get available classes from directory structure
        self.available_classes = self._get_available_classes()
        
        # Limit classes if specified
        if max_classes:
            self.available_classes = self.available_classes[:max_classes]
        
        print(f"Found {len(self.available_classes)} classes in {split} split")
        
        # Create dataset samples
        self.samples = []
        self._load_samples()
        
        # Pre-compute CLIP embeddings for all classes
        self.text_embeddings = self._compute_text_embeddings()
        
    def _load_class_mappings(self) -> Dict[str, str]:
        """Load ImageNet class ID to human-readable name mappings from words.txt"""
        words_file = self.root_dir / "words.txt"
        class_mappings = {}
        
        if words_file.exists():
            with open(words_file, 'r') as f:
                for line in f:
                    if '\t' in line:
                        class_id, class_name = line.strip().split('\t', 1)
                        # Clean up class name - take first name if there are multiple
                        clean_name = class_name.split(',')[0].strip()
                        class_mappings[class_id] = clean_name
        
        print(f"Loaded {len(class_mappings)} class mappings from words.txt")
        return class_mappings
    
    def _get_available_classes(self) -> List[str]:
        """Get list of available class directories"""
        split_dir = self.root_dir / self.split
        
        if not split_dir.exists():
            raise ValueError(f"Split directory {split_dir} not found")
        
        # Get all class directories (starting with 'n')
        class_dirs = [d.name for d in split_dir.iterdir() 
                     if d.is_dir() and d.name.startswith('n')]
        
        # Sort for consistency
        class_dirs.sort()
        
        return class_dirs
    
    def _load_samples(self):
        """Load image samples for available classes"""
        split_dir = self.root_dir / self.split
        
        for class_id in self.available_classes:
            class_dir = split_dir / class_id
            if not class_dir.exists():
                print(f"Warning: Class directory {class_dir} not found")
                continue
            
            # Load all JPEG images
            image_files = list(class_dir.glob('*.JPEG')) + list(class_dir.glob('*.jpg'))
            
            for img_path in image_files:
                self.samples.append((str(img_path), class_id))
        
        print(f"Loaded {len(self.samples)} samples from {len(self.available_classes)} classes")
    
    def _compute_text_embeddings(self) -> Dict[str, torch.Tensor]:
        """Pre-compute CLIP text embeddings for all classes"""
        embeddings = {}
        device = next(self.clip_model.parameters()).device
        
        print("Computing CLIP text embeddings...")
        
        with torch.no_grad():
            for class_id in tqdm(self.available_classes, desc="Computing embeddings"):
                class_name = self.class_to_name.get(class_id, f"class {class_id}")
                
                # Create descriptive text prompt
                text_prompt = f"a photo of a {class_name}"
                
                # Tokenize and encode text
                text_tokens = clip.tokenize([text_prompt]).to(device)
                text_embedding = self.clip_model.encode_text(text_tokens)
                text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
                
                # Convert to float32 and store
                embeddings[class_id] = text_embedding.cpu().squeeze(0).float()
        
        return embeddings
    
    def get_class_info(self, class_id: str) -> Dict[str, str]:
        """Get human-readable information about a class"""
        class_name = self.class_to_name.get(class_id, "unknown")
        return {
            'class_id': class_id,
            'class_name': class_name,
            'text_prompt': f"a photo of a {class_name}"
        }
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, class_id = self.samples[idx]
        
        try:
            # Load image
            image = Image.open(img_path).convert('RGB')
            
            if self.transform:
                image = self.transform(image)
            
            # Get text embedding and convert to float32
            text_embedding = self.text_embeddings[class_id].float()
            class_name = self.class_to_name.get(class_id, f"class {class_id}")
            
            return image, text_embedding, class_name
            
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a random sample instead
            return self.__getitem__((idx + 1) % len(self.samples))

def create_imagenet_mini_loaders(root_dir: str, clip_model, batch_size: int = 4, 
                                image_size: int = 256, num_workers: int = 4,
                                max_classes: Optional[int] = None):
    """Create training and validation data loaders for ImageNet Mini"""
    
    # Training transforms
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # [-1, 1]
    ])
    
    # Validation transforms
    val_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    print(f"Creating ImageNet Mini datasets from {root_dir}")
    
    train_dataset = ImageNetMiniDataset(
        root_dir, clip_model, train_transform, 'train', max_classes
    )
    
    # Check if val split exists, otherwise use train split for validation
    val_split = 'val' if (Path(root_dir) / 'val').exists() else 'train'
    if val_split == 'train':
        print("No separate validation split found, using training data for validation")
    
    val_dataset = ImageNetMiniDataset(
        root_dir, clip_model, val_transform, val_split, max_classes
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, 
        num_workers=num_workers, pin_memory=True, drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, 
        num_workers=num_workers, pin_memory=True
    )
    
    print(f"Created data loaders:")
    print(f"  Train: {len(train_loader)} batches ({len(train_dataset)} samples)")
    print(f"  Val: {len(val_loader)} batches ({len(val_dataset)} samples)")
    
    return train_loader, val_loader, train_dataset.available_classes

# =============================================================================
# TRAINING FUNCTIONS
# =============================================================================

def compute_loss(model: nn.Module, x_0: torch.Tensor, text_embeddings: torch.Tensor, 
                scheduler: DDIMScheduler, device: torch.device) -> torch.Tensor:
    """Compute diffusion training loss"""
    batch_size = x_0.size(0)
    
    # Sample random timesteps
    timesteps = torch.randint(0, scheduler.num_train_timesteps, (batch_size,), device=device).long()
    
    # Sample noise
    noise = torch.randn_like(x_0)
    
    # Add noise to images
    noisy_images = scheduler.add_noise(x_0, noise, timesteps)
    
    # FIX: Convert text embeddings to same dtype as model
    text_embeddings = text_embeddings.float()
    
    # Predict noise
    predicted_noise = model(noisy_images, timesteps, text_embeddings)
    
    # Compute loss
    loss = F.mse_loss(predicted_noise, noise)
    
    return loss

def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                scheduler: DDIMScheduler, num_epochs: int, device: torch.device,
                save_dir: Path, learning_rate: float = 1e-4):
    """Training loop"""
    
    save_dir.mkdir(parents=True, exist_ok=True)
    writer = SummaryWriter(save_dir / "logs")
    
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-6)
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs * len(train_loader), eta_min=1e-6
    )
    
    global_step = 0
    best_val_loss = float('inf')
    
    print(f"Starting training on {device}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    for epoch in range(num_epochs):
        model.train()
        epoch_losses = []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}")
        
        for batch_idx, (images, text_embeddings, class_names) in enumerate(pbar):
            images = images.to(device)
            text_embeddings = text_embeddings.to(device)
            
            # Compute loss
            loss = compute_loss(model, images, text_embeddings, scheduler, device)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            lr_scheduler.step()
            
            epoch_losses.append(loss.item())
            
            if global_step % 100 == 0:
                writer.add_scalar('Train/Loss', loss.item(), global_step)
                writer.add_scalar('Train/LR', optimizer.param_groups[0]['lr'], global_step)
                
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'lr': f"{optimizer.param_groups[0]['lr']:.2e}"
                })
            
            global_step += 1
        
        # Validation
        model.eval()
        val_losses = []
        
        with torch.no_grad():
            for images, text_embeddings, class_names in val_loader:
                images = images.to(device)
                text_embeddings = text_embeddings.to(device)
                
                val_loss = compute_loss(model, images, text_embeddings, scheduler, device)
                val_losses.append(val_loss.item())
        
        avg_train_loss = np.mean(epoch_losses)
        avg_val_loss = np.mean(val_losses)
        
        writer.add_scalar('Epoch/TrainLoss', avg_train_loss, epoch)
        writer.add_scalar('Epoch/ValLoss', avg_val_loss, epoch)
        
        print(f"Epoch {epoch}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_path = save_dir / "best_model.pth"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_loss': best_val_loss,
                'model_config': {
                    'text_embed_dim': 512,
                    'model_channels': 128,
                    'image_size': 256,
                    'in_channels': 3
                }
            }, best_model_path)
    
    # Save final model
    final_model_path = save_dir / "ddim_imagenet.pth"
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': num_epochs,
        'val_loss': best_val_loss,
        'model_config': {
            'text_embed_dim': 512,
            'model_channels': 128,
            'image_size': 256,
            'in_channels': 3
        }
    }, final_model_path)
    
    writer.close()
    return str(final_model_path)

# =============================================================================
# DDIM SAMPLING AND DEMO
# =============================================================================

@torch.no_grad()
def ddim_sample(model: nn.Module, scheduler: DDIMScheduler, text_embeddings: torch.Tensor,
                num_inference_steps: int = 20, eta: float = 0.0, cfg_scale: float = 7.5,
                device: torch.device = None) -> torch.Tensor:
    """Generate images using DDIM sampling"""
    
    batch_size = text_embeddings.size(0)
    image_shape = (batch_size, 3, 256, 256)
    
    # For classifier-free guidance
    if cfg_scale > 1.0:
        # Create unconditional embedding (zeros)
        uncond_embeddings = torch.zeros_like(text_embeddings)
        text_embeddings = torch.cat([uncond_embeddings, text_embeddings], dim=0)
        
    # Initialize with random noise
    if cfg_scale > 1.0:
        images = torch.randn((batch_size * 2, 3, 256, 256), device=device)
    else:
        images = torch.randn(image_shape, device=device)
    
    # Set timesteps
    scheduler.set_timesteps(num_inference_steps, device)
    
    model.eval()
    
    for i, t in enumerate(tqdm(scheduler.timesteps, desc="DDIM Sampling")):
        # Expand timestep to batch dimension
        timestep_batch = t.expand(images.shape[0])
        
        # Predict noise
        noise_pred = model(images, timestep_batch, text_embeddings.repeat(images.shape[0] // text_embeddings.shape[0], 1))
        
        # Apply classifier-free guidance
        if cfg_scale > 1.0:
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + cfg_scale * (noise_pred_text - noise_pred_uncond)
            images = images[:batch_size]  # Keep only conditional batch
        
        # DDIM step
        images = scheduler.step(noise_pred, t.item(), images, eta=eta)
        
        # Update images for next iteration if using CFG
        if cfg_scale > 1.0:
            images = torch.cat([images, images], dim=0)
    
    # Return only the conditional batch
    if cfg_scale > 1.0:
        images = images[:batch_size]
    
    return images

class DDIMDemo:
    """Interactive demo for DDIM text-to-image generation"""
    
    def __init__(self, model_path: str, device: str = "auto"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device == "auto" else torch.device(device)
        
        # Load CLIP model
        print("Loading CLIP model...")
        self.clip_model, _ = clip.load("ViT-B/32", device=self.device)
        self.clip_model.eval()
        
        # Load diffusion model
        print(f"Loading diffusion model from {model_path}...")
        checkpoint = torch.load(model_path, map_location=self.device)
        
        model_config = checkpoint.get('model_config', {
            'text_embed_dim': 512,
            'model_channels': 128,
            'image_size': 256,
            'in_channels': 3
        })
        
        self.model = TextConditionalUNet(**model_config).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        # Initialize DDIM scheduler
        # Create DDIM scheduler
        scheduler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="linear")
        scheduler = scheduler.to(device)  # Move scheduler tensors to GPU

        print("DDIM Demo initialized successfully!")
    
    def encode_text(self, text_prompt: str) -> torch.Tensor:
        """Encode text prompt using CLIP"""
        with torch.no_grad():
            text_tokens = clip.tokenize([text_prompt]).to(self.device)
            text_embedding = self.clip_model.encode_text(text_tokens)
            text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
        return text_embedding
    
    def generate_image(self, text_prompt: str, num_inference_steps: int = 20,
                      eta: float = 0.0, cfg_scale: float = 7.5, 
                      seed: Optional[int] = None) -> torch.Tensor:
        """Generate image from text prompt using DDIM"""
        
        if seed is not None:
            torch.manual_seed(seed)
            np.random.seed(seed)
        
        # Encode text
        text_embedding = self.encode_text(text_prompt)
        
        # Generate image
        images = ddim_sample(
            self.model, self.scheduler, text_embedding,
            num_inference_steps=num_inference_steps,
            eta=eta, cfg_scale=cfg_scale, device=self.device
        )
        
        return images[0]  # Return first image
    
    def denormalize_image(self, image: torch.Tensor) -> torch.Tensor:
        """Convert from [-1, 1] to [0, 1] range"""
        return torch.clamp((image + 1) / 2, 0, 1)
    
    def tensor_to_pil(self, image_tensor: torch.Tensor) -> Image.Image:
        """Convert tensor to PIL Image"""
        image_tensor = self.denormalize_image(image_tensor)
        image_np = image_tensor.cpu().permute(1, 2, 0).numpy()
        image_pil = Image.fromarray((image_np * 255).astype(np.uint8))
        return image_pil
    
    def compute_clip_similarity(self, generated_image: torch.Tensor, text_prompt: str) -> float:
        """Compute CLIP similarity between generated image and text prompt"""
        
        # Prepare image for CLIP
        image = self.denormalize_image(generated_image)
        
        clip_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], 
                               std=[0.26862954, 0.26130258, 0.27577711])
        ])
        
        image_input = clip_transform(image).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            # Get image and text features
            image_features = self.clip_model.encode_image(image_input)
            text_features = self.encode_text(text_prompt)
            
            # Normalize and compute similarity
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            
            similarity = torch.cosine_similarity(image_features, text_features).item()
        
        return similarity

def generate_image_demo(
    model_path: str = "checkpoints/ddim_imagenet.pth",
    text_prompt: str = "a photo of a golden retriever",
    output_dir: str = "demo_outputs/",
    device: str = "auto",
    cfg_scale: float = 7.5,
    num_inference_steps: int = 20,
    eta: float = 0.0,
    seed: Optional[int] = None
) -> Dict[str, Any]:
    """
    DDIM demo function for fast text-to-image generation
    
    Args:
        model_path: Path to trained model checkpoint
        text_prompt: Text description for image generation
        output_dir: Directory to save outputs
        device: Device to use ('auto', 'cuda', 'cpu')
        cfg_scale: Classifier-free guidance scale (1.0 = no guidance)
        num_inference_steps: Number of DDIM denoising steps (10-50)
        eta: DDIM sampling noise factor (0.0 = deterministic, 1.0 = stochastic)
        seed: Random seed for reproducibility
    
    Returns:
        Dictionary with generated image and metadata
    """
    
    # Setup output directory
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Initialize demo
    demo = DDIMDemo(model_path, device)
    
    # Generate image
    print(f"\nGenerating image for prompt: '{text_prompt}'")
    print(f"CFG Scale: {cfg_scale}, Steps: {num_inference_steps}, eta: {eta}")
    if seed is not None:
        print(f"Seed: {seed}")
    
    start_time = time.time()
    generated_image = demo.generate_image(
        text_prompt=text_prompt,
        num_inference_steps=num_inference_steps,
        eta=eta,
        cfg_scale=cfg_scale,
        seed=seed
    )
    generation_time = time.time() - start_time
    
    # Compute CLIP similarity
    clip_similarity = demo.compute_clip_similarity(generated_image, text_prompt)
    
    # Create safe filename
    safe_prompt = "".join(c for c in text_prompt if c.isalnum() or c in (' ', '-', '_')).rstrip()
    safe_prompt = safe_prompt.replace(' ', '_')[:50]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{safe_prompt}_{timestamp}.png"
    
    # Save image
    final_pil = demo.tensor_to_pil(generated_image)
    image_path = output_path / filename
    final_pil.save(image_path)
    
    # Save metadata
    metadata = {
        'text_prompt': text_prompt,
        'clip_similarity': clip_similarity,
        'cfg_scale': cfg_scale,
        'num_inference_steps': num_inference_steps,
        'eta': eta,
        'seed': seed,
        'generation_time': generation_time,
        'model_path': model_path,
        'timestamp': timestamp
    }
    
    metadata_path = output_path / f"{safe_prompt}_{timestamp}_metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"\nGeneration completed in {generation_time:.2f} seconds")
    print(f"CLIP Similarity Score: {clip_similarity:.3f}")
    print(f"Image saved to: {image_path}")
    print(f"Metadata saved to: {metadata_path}")
    
    return {
        'generated_image': generated_image,
        'clip_similarity': clip_similarity,
        'generation_time': generation_time,
        'metadata': metadata,
        'image_path': str(image_path),
        'metadata_path': str(metadata_path)
    }

# =============================================================================
# MAIN FUNCTIONS
# =============================================================================

def main_train():
    """Main training function"""
    
    # Configuration
    config = {
        'batch_size': 2,  # 🔥 REDUCED from 8 to 2 for RTX 4060
        'learning_rate': 1e-4,  # 🔥 REDUCED learning rate to compensate for smaller batch
        'num_epochs': 40,  # 🔥 INCREASED epochs slightly since smaller batch size
        'image_size': 256,
        'save_dir': Path('./checkpoints'),
        'imagenet_root': r'D:\DDPM-diffusion\data\imagenet-mini',
        'max_classes': 30,  # 🔥 FURTHER REDUCED for memory efficiency
        'num_workers': 0,  # No multiprocessing for stability
        'gradient_clip': 1.0,
        'save_every': 5,  # Save less frequently to save disk space
        'demo_every': 10,   
        'guidance_scale': 7.5,
        # RTX 4060 Memory optimizations
        'pin_memory': False,  # 🔥 DISABLED to save GPU memory
        'persistent_workers': False,
        'prefetch_factor': 1,  # 🔥 REDUCED prefetch
        'gradient_accumulation_steps': 4,  # 🔥 NEW: Simulate larger batch via accumulation
    }
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load CLIP model
    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    
    # Create data loaders for ImageNet Mini
    train_loader, val_loader, available_classes = create_imagenet_mini_loaders(
        root_dir=config['imagenet_root'],
        clip_model=clip_model,
        batch_size=config['batch_size'],
        image_size=config['image_size'],
        num_workers=config['num_workers'],
        max_classes=config.get('max_classes', None)
    )
    
    # Create model
    model = TextConditionalUNet(
        text_embed_dim=512,
        model_channels=128,
        image_size=config['image_size'],
        in_channels=3
    ).to(device)
    
    # Create DDIM scheduler
    scheduler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="linear")
    
    # Train the model
    final_model_path = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        scheduler=scheduler,
        num_epochs=config['num_epochs'],
        device=device,
        save_dir=config['save_dir'],
        learning_rate=config['learning_rate']
    )
    
    print(f"Training completed! Model saved to: {final_model_path}")

def main_demo():
    """Main demo function"""
    
    parser = argparse.ArgumentParser(description="DDIM Text-to-Image Demo")
    parser.add_argument("--model_path", type=str, default="checkpoints/ddim_imagenet.pth",
                       help="Path to trained model checkpoint")
    parser.add_argument("--prompt", type=str, default="a photo of a golden retriever",
                       help="Text prompt for image generation")
    parser.add_argument("--output_dir", type=str, default="demo_outputs/",
                       help="Output directory for generated images")
    parser.add_argument("--device", type=str, default="auto", choices=["auto", "cuda", "cpu"],
                       help="Device to use for generation")
    parser.add_argument("--cfg_scale", type=float, default=7.5,
                       help="Classifier-free guidance scale")
    parser.add_argument("--steps", type=int, default=20,
                       help="Number of DDIM denoising steps")
    parser.add_argument("--eta", type=float, default=0.0,
                       help="DDIM sampling noise factor (0.0=deterministic)")
    parser.add_argument("--seed", type=int, default=None,
                       help="Random seed for reproducibility")
    parser.add_argument("--batch", nargs="+", type=str,
                       help="Generate multiple prompts in batch")
    
    args = parser.parse_args()
    
    if args.batch:
        # Batch generation
        for prompt in args.batch:
            generate_image_demo(
                model_path=args.model_path,
                text_prompt=prompt,
                output_dir=args.output_dir,
                device=args.device,
                cfg_scale=args.cfg_scale,
                num_inference_steps=args.steps,
                eta=args.eta,
                seed=args.seed
            )
    else:
        # Single generation
        generate_image_demo(
            model_path=args.model_path,
            text_prompt=args.prompt,
            output_dir=args.output_dir,
            device=args.device,
            cfg_scale=args.cfg_scale,
            num_inference_steps=args.steps,
            eta=args.eta,
            seed=args.seed
        )

def test_dataset():
    """Test ImageNet Mini dataset loading"""
    print("Testing ImageNet Mini dataset loading...")
    
    # Load CLIP model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    
    # Test data loader creation
    dataset_path = r'D:\DDPM-diffusion\data\imagenet-mini'
    
    try:
        train_loader, val_loader, available_classes = create_imagenet_mini_loaders(
            root_dir=dataset_path,
            clip_model=clip_model,
            batch_size=2,
            image_size=256,
            num_workers=0,  # Use 0 for testing
            max_classes=10  # Test with just 10 classes
        )
        
        print(f"\n✅ Dataset loading successful!")
        print(f"Available classes (first 10): {available_classes[:10]}")
        
        # Test loading a batch
        print("\nTesting batch loading...")
        for batch_idx, (images, text_embeddings, class_names) in enumerate(train_loader):
            print(f"  Batch {batch_idx + 1}:")
            print(f"    Images shape: {images.shape}")
            print(f"    Text embeddings shape: {text_embeddings.shape}")
            print(f"    Class names: {class_names}")
            
            if batch_idx >= 2:  # Test just a few batches
                break
        
        print("\n✅ Dataset test completed successfully!")
        return True
        
    except Exception as e:
        print(f"\n❌ Dataset test failed: {e}")
        import traceback
        traceback.print_exc()
        return False

In [2]:
def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                scheduler: DDIMScheduler, num_epochs: int, device: torch.device,
                save_dir: Path, learning_rate: float = 1e-4, clip_model=None):
    """Enhanced training loop with comprehensive visualization"""
    
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Initialize TensorBoard writer if available
    if TENSORBOARD_AVAILABLE:
        writer = SummaryWriter(save_dir / "logs")
        print("📊 TensorBoard logging enabled")
    else:
        writer = None
        print("⚠️ TensorBoard not available - using basic logging")
    
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-6)
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs * len(train_loader), eta_min=1e-6
    )
    
    # Training configuration
    SAVE_SAMPLES_EVERY = 1000      # Generate samples every 1000 steps
    SAVE_CHECKPOINT_EVERY = 5      # Save checkpoint every 5 epochs
    PLOT_PROGRESS_EVERY = 2        # Plot progress every 2 epochs
    SAMPLE_INFERENCE_STEPS = 10    # Fast sampling for training visualization
    
    # Training state
    global_step = 0
    best_val_loss = float('inf')
    losses_history = {
        'train_loss': [],
        'val_loss': [],
        'steps': [],
        'epochs': []
    }
    
    # Sample prompts for consistent visualization
    sample_prompts = [
        "a photo of a golden retriever dog",
        "a photo of a tabby cat sitting", 
        "a photo of a red sports car",
        "a photo of a beautiful sunflower",
        "a photo of a majestic eagle flying",
        "a photo of a tropical beach sunset"
    ]
    
    print(f"🚀 Starting enhanced training on {device}")
    print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"🎨 Will generate samples every {SAVE_SAMPLES_EVERY} steps")
    print(f"💾 Will save checkpoints every {SAVE_CHECKPOINT_EVERY} epochs")
    
    for epoch in range(num_epochs):
        model.train()
        epoch_losses = []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, (images, text_embeddings, class_names) in enumerate(pbar):
            images = images.to(device)
            text_embeddings = text_embeddings.to(device)
            
            # Compute loss
            loss = compute_loss(model, images, text_embeddings, scheduler, device)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            lr_scheduler.step()
            
            epoch_losses.append(loss.item())
            losses_history['train_loss'].append(loss.item())
            losses_history['steps'].append(global_step)
            
            # Logging
            current_lr = optimizer.param_groups[0]['lr']
            if global_step % 100 == 0:
                if writer:
                    writer.add_scalar('Train/Loss', loss.item(), global_step)
                    writer.add_scalar('Train/LR', current_lr, global_step)
                    writer.add_scalar('Train/GradNorm', 
                                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float('inf')), 
                                    global_step)
                
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'lr': f"{current_lr:.2e}",
                    'step': global_step
                })
            
            # Generate training samples periodically
            if clip_model and global_step % SAVE_SAMPLES_EVERY == 0 and global_step > 0:
                try:
                    save_training_samples(
                        model=model,
                        clip_model=clip_model, 
                        scheduler=scheduler,
                        device=device,
                        save_dir=save_dir,
                        global_step=global_step,
                        sample_prompts=sample_prompts,
                        num_inference_steps=SAMPLE_INFERENCE_STEPS
                    )
                except Exception as e:
                    print(f"⚠️ Failed to generate samples at step {global_step}: {e}")
            
            global_step += 1
        
        # Validation after each epoch
        model.eval()
        val_losses = []
        
        print(f"\n🔍 Running validation for epoch {epoch+1}...")
        with torch.no_grad():
            for val_batch_idx, (images, text_embeddings, class_names) in enumerate(tqdm(val_loader, desc="Validation")):
                images = images.to(device)
                text_embeddings = text_embeddings.to(device)
                
                val_loss = compute_loss(model, images, text_embeddings, scheduler, device)
                val_losses.append(val_loss.item())
                
                # Limit validation batches for speed
                if val_batch_idx >= 50:  # Only validate on first 50 batches
                    break
        
        # Calculate averages
        avg_train_loss = np.mean(epoch_losses)
        avg_val_loss = np.mean(val_losses)
        
        losses_history['val_loss'].append(avg_val_loss)
        losses_history['epochs'].append(epoch)
        
        # Log to TensorBoard
        if writer:
            writer.add_scalar('Epoch/TrainLoss', avg_train_loss, epoch)
            writer.add_scalar('Epoch/ValLoss', avg_val_loss, epoch)
            writer.add_scalar('Epoch/LearningRate', optimizer.param_groups[0]['lr'], epoch)
        
        print(f"\n📊 Epoch {epoch+1} Results:")
        print(f"   Train Loss: {avg_train_loss:.4f}")
        print(f"   Val Loss: {avg_val_loss:.4f}")
        print(f"   Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Save best model
        is_best = avg_val_loss < best_val_loss
        if is_best:
            best_val_loss = avg_val_loss
            save_model_checkpoint_with_metadata(
                model=model,
                optimizer=optimizer,
                scheduler_lr=lr_scheduler,
                epoch=epoch,
                global_step=global_step,
                train_loss=avg_train_loss,
                val_loss=avg_val_loss,
                save_dir=save_dir,
                is_best=True,
                losses_history=losses_history
            )
        
        # Regular checkpoint saving
        if (epoch + 1) % SAVE_CHECKPOINT_EVERY == 0:
            save_model_checkpoint_with_metadata(
                model=model,
                optimizer=optimizer,
                scheduler_lr=lr_scheduler,
                epoch=epoch,
                global_step=global_step,
                train_loss=avg_train_loss,
                val_loss=avg_val_loss,
                save_dir=save_dir,
                is_best=False,
                losses_history=losses_history
            )
        
        # Plot training progress
        if (epoch + 1) % PLOT_PROGRESS_EVERY == 0:
            try:
                plot_training_progress(save_dir, losses_history, epoch, avg_val_loss)
            except Exception as e:
                print(f"⚠️ Failed to plot progress: {e}")
        
        # Generate final samples for this epoch (if it's a milestone)
        if clip_model and (epoch + 1) % 5 == 0:
            try:
                save_training_samples(
                    model=model,
                    clip_model=clip_model,
                    scheduler=scheduler, 
                    device=device,
                    save_dir=save_dir,
                    global_step=f"epoch_{epoch+1:03d}",
                    sample_prompts=sample_prompts,
                    num_inference_steps=20  # Higher quality for epoch milestones
                )
            except Exception as e:
                print(f"⚠️ Failed to generate epoch samples: {e}")
    
    # Save final model
    final_model_path = save_dir / "ddim_imagenet_final.pth"
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': num_epochs,
        'global_step': global_step,
        'val_loss': best_val_loss,
        'losses_history': losses_history,
        'model_config': {
            'text_embed_dim': 512,
            'model_channels': 128,
            'image_size': 256,
            'in_channels': 3
        },
        'training_metadata': {
            'total_epochs': num_epochs,
            'best_val_loss': best_val_loss,
            'final_train_loss': avg_train_loss,
            'timestamp': datetime.now().isoformat(),
        }
    }, final_model_path)
    
    # Close TensorBoard writer
    if writer:
        writer.close()
    
    # Create final training summary
    create_training_summary(save_dir, losses_history, num_epochs, best_val_loss)
    
    print(f"\n🎉 Training completed!")
    print(f"   Final model saved to: {final_model_path}")
    print(f"   Best validation loss: {best_val_loss:.4f}")
    print(f"   Total training steps: {global_step}")
    print(f"   Check visualization files in: {save_dir}")
    
    return str(final_model_path)

def create_training_summary(save_dir: Path, losses_history: Dict, 
                          num_epochs: int, best_val_loss: float):
    """Create a comprehensive training summary with visualizations"""
    
    summary_dir = save_dir / "training_summary"
    summary_dir.mkdir(exist_ok=True)
    
    # Create comprehensive loss plot
    plt.figure(figsize=(15, 5))
    
    # Training loss over steps
    plt.subplot(1, 3, 1)
    if losses_history['train_loss']:
        plt.plot(losses_history['steps'], losses_history['train_loss'], 
                alpha=0.3, color='blue', linewidth=0.5)
        # Add smoothed line
        window_size = max(1, len(losses_history['train_loss']) // 100)
        smoothed_loss = np.convolve(losses_history['train_loss'], 
                                  np.ones(window_size)/window_size, mode='valid')
        smoothed_steps = losses_history['steps'][:len(smoothed_loss)]
        plt.plot(smoothed_steps, smoothed_loss, color='blue', linewidth=2, 
                label=f'Training Loss (smoothed)')
        plt.title('Training Loss Over Steps')
        plt.xlabel('Training Step')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    # Validation loss over epochs
    plt.subplot(1, 3, 2)
    if losses_history['val_loss']:
        plt.plot(losses_history['epochs'], losses_history['val_loss'], 
                'ro-', markersize=4, label='Validation Loss')
        plt.axhline(y=best_val_loss, color='red', linestyle='--', alpha=0.7, 
                   label=f'Best: {best_val_loss:.4f}')
        plt.title('Validation Loss Over Epochs')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    # Loss comparison
    plt.subplot(1, 3, 3)
    if losses_history['train_loss'] and losses_history['val_loss']:
        # Sample training loss at epoch intervals
        epoch_train_losses = []
        for epoch in losses_history['epochs']:
            # Find training losses around this epoch
            epoch_steps = epoch * (len(losses_history['steps']) // num_epochs)
            start_idx = max(0, epoch_steps - 50)
            end_idx = min(len(losses_history['train_loss']), epoch_steps + 50)
            if start_idx < end_idx:
                epoch_train_losses.append(np.mean(losses_history['train_loss'][start_idx:end_idx]))
            else:
                epoch_train_losses.append(losses_history['train_loss'][-1])
        
        plt.plot(losses_history['epochs'], epoch_train_losses, 'b-', 
                label='Training Loss', alpha=0.7)
        plt.plot(losses_history['epochs'], losses_history['val_loss'], 'r-', 
                label='Validation Loss', alpha=0.7)
        plt.title('Training vs Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    summary_path = summary_dir / "training_summary.png"
    plt.savefig(summary_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save numerical summary
    summary_text = f"""
# Training Summary

## Configuration
- Total Epochs: {num_epochs}
- Total Steps: {len(losses_history['train_loss'])}
- Best Validation Loss: {best_val_loss:.6f}

## Final Metrics
- Final Training Loss: {losses_history['train_loss'][-1] if losses_history['train_loss'] else 'N/A':.6f}
- Final Validation Loss: {losses_history['val_loss'][-1] if losses_history['val_loss'] else 'N/A':.6f}

## Training Progress
- Training loss improved from {losses_history['train_loss'][0]:.6f} to {losses_history['train_loss'][-1]:.6f}
- Validation loss improved from {losses_history['val_loss'][0]:.6f} to {min(losses_history['val_loss']):.6f}

## Files Generated
- Model checkpoints: checkpoints/
- Training samples: training_samples/
- Progress plots: training_plots/
- TensorBoard logs: logs/
"""
    
    with open(summary_dir / "training_summary.txt", 'w') as f:
        f.write(summary_text)
    
    print(f"📋 Training summary saved to {summary_dir}")

In [3]:
# =============================================================================
# MISSING VISUALIZATION FUNCTIONS
# =============================================================================

def tensor_to_pil(image_tensor: torch.Tensor) -> Image.Image:
    """Convert tensor to PIL Image for visualization"""
    # Denormalize from [-1, 1] to [0, 1]
    image_tensor = torch.clamp((image_tensor + 1) / 2, 0, 1)
    image_np = image_tensor.cpu().permute(1, 2, 0).numpy()
    image_pil = Image.fromarray((image_np * 255).astype(np.uint8))
    return image_pil

def save_training_samples(model: nn.Module, clip_model, scheduler: DDIMScheduler, 
                         device: torch.device, save_dir: Path, global_step: int,
                         sample_prompts: List[str] = None, num_inference_steps: int = 10):
    """Generate and save sample images during training to visualize progress"""
    
    if sample_prompts is None:
        sample_prompts = [
            "a photo of a golden retriever dog",
            "a photo of a tabby cat sitting",
            "a photo of a red sports car", 
            "a photo of a beautiful sunflower",
            "a photo of a majestic eagle flying",
            "a photo of a tropical beach sunset"
        ]
    
    model.eval()
    sample_dir = save_dir / "training_samples"
    sample_dir.mkdir(exist_ok=True)
    
    print(f"\n🎨 Generating training samples at step {global_step}...")
    
    generated_images = []
    
    with torch.no_grad():
        for i, prompt in enumerate(sample_prompts):
            try:
                # Encode text with CLIP
                text_tokens = clip.tokenize([prompt]).to(device)
                text_embedding = clip_model.encode_text(text_tokens)
                text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
                
                # Quick DDIM generation for speed during training
                generated_img = ddim_sample(
                    model=model, 
                    scheduler=scheduler, 
                    text_embeddings=text_embedding,
                    num_inference_steps=num_inference_steps,  # Fast generation
                    cfg_scale=7.5, 
                    eta=0.0,  # Deterministic
                    device=device
                )[0]  # Get first image from batch
                
                # Convert to PIL and save
                img_pil = tensor_to_pil(generated_img)
                
                # Create descriptive filename
                safe_prompt = "".join(c for c in prompt if c.isalnum() or c in (' ', '-', '_')).strip()
                safe_prompt = safe_prompt.replace(' ', '_')[:30]
                
                img_filename = f"step_{global_step:06d}_prompt_{i:02d}_{safe_prompt}.png"
                img_path = sample_dir / img_filename
                img_pil.save(img_path)
                
                generated_images.append((prompt, img_pil))
                
                print(f"  ✅ Generated: {prompt[:40]}...")
                
            except Exception as e:
                print(f"  ❌ Failed to generate for '{prompt}': {e}")
                continue
    
    # Create a grid visualization
    if generated_images:
        try:
            create_sample_grid(generated_images, sample_dir, global_step)
        except Exception as e:
            print(f"  ⚠️ Failed to create grid: {e}")
    
    model.train()  # Switch back to training mode
    print(f"  💾 Saved {len(generated_images)} samples to {sample_dir}")

def create_sample_grid(generated_images: List[Tuple[str, Image.Image]], 
                      save_dir: Path, global_step: int):
    """Create a grid visualization of generated samples"""
    
    if not generated_images:
        return
    
    # Calculate grid size
    num_images = len(generated_images)
    grid_cols = min(3, num_images)
    grid_rows = (num_images + grid_cols - 1) // grid_cols
    
    # Create figure
    fig_width = grid_cols * 4
    fig_height = grid_rows * 4.5  # Extra space for text
    
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(fig_width, fig_height))
    fig.suptitle(f'Training Samples - Step {global_step}', fontsize=16, fontweight='bold')
    
    # Handle single image case
    if grid_rows == 1 and grid_cols == 1:
        axes = [axes]
    elif grid_rows == 1 or grid_cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    # Plot images
    for i, (prompt, img_pil) in enumerate(generated_images):
        if i < len(axes):
            axes[i].imshow(img_pil)
            axes[i].set_title(prompt[:40] + "..." if len(prompt) > 40 else prompt, 
                            fontsize=10, wrap=True)
            axes[i].axis('off')
    
    # Hide empty subplots
    for i in range(len(generated_images), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    
    # Save grid
    grid_filename = f"training_grid_step_{global_step:06d}.png"
    grid_path = save_dir / grid_filename
    plt.savefig(grid_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"  📊 Saved sample grid to {grid_path}")

def plot_training_progress(save_dir: Path, losses: Dict[str, List[float]], 
                          epoch: int, val_loss: float = None):
    """Plot and save training loss curves"""
    
    plots_dir = save_dir / "training_plots"
    plots_dir.mkdir(exist_ok=True)
    
    plt.figure(figsize=(12, 4))
    
    # Plot training loss
    plt.subplot(1, 2, 1)
    if 'train_loss' in losses and losses['train_loss']:
        plt.plot(losses['train_loss'], label='Training Loss', color='blue', alpha=0.7)
        plt.title('Training Loss')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    # Plot validation loss if available
    plt.subplot(1, 2, 2)
    if 'val_loss' in losses and losses['val_loss']:
        epochs = list(range(len(losses['val_loss'])))
        plt.plot(epochs, losses['val_loss'], label='Validation Loss', 
                color='red', marker='o', markersize=4)
        if val_loss is not None:
            plt.axhline(y=val_loss, color='red', linestyle='--', alpha=0.5, 
                       label=f'Current: {val_loss:.4f}')
        plt.title('Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    plot_filename = f"training_progress_epoch_{epoch:03d}.png"
    plot_path = plots_dir / plot_filename
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"  📈 Saved training progress plot to {plot_path}")

def save_model_checkpoint_with_metadata(model: nn.Module, optimizer, scheduler_lr, 
                                       epoch: int, global_step: int, train_loss: float,
                                       val_loss: float, save_dir: Path, 
                                       is_best: bool = False, losses_history: Dict = None):
    """Save model checkpoint with comprehensive metadata"""
    
    checkpoint_data = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'lr_scheduler_state_dict': scheduler_lr.state_dict(),
        'epoch': epoch,
        'global_step': global_step,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'model_config': {
            'text_embed_dim': 512,
            'model_channels': 128,
            'image_size': 256,
            'in_channels': 3
        },
        'training_metadata': {
            'timestamp': datetime.now().isoformat(),
            'total_parameters': sum(p.numel() for p in model.parameters()),
            'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
        }
    }
    
    # Add loss history if provided
    if losses_history:
        checkpoint_data['losses_history'] = losses_history
    
    # Save checkpoint
    if is_best:
        checkpoint_path = save_dir / "best_model.pth"
        print(f"  🥇 Saving BEST model checkpoint (val_loss: {val_loss:.4f})")
    else:
        checkpoint_path = save_dir / f"checkpoint_epoch_{epoch:03d}.pth"
        print(f"  💾 Saving checkpoint for epoch {epoch}")
    
    torch.save(checkpoint_data, checkpoint_path)
    
    # Always save as latest for easy resuming
    latest_path = save_dir / "latest_checkpoint.pth"
    torch.save(checkpoint_data, latest_path)
    
    return str(checkpoint_path)

In [4]:
#test the dataset loading
test_dataset()

Testing ImageNet Mini dataset loading...
Creating ImageNet Mini datasets from D:\DDPM-diffusion\data\imagenet-mini
Loaded 82115 class mappings from words.txt
Found 10 classes in train split
Loaded 316 samples from 10 classes
Computing CLIP text embeddings...


Computing embeddings: 100%|██████████| 10/10 [00:00<00:00, 48.11it/s]


Loaded 82115 class mappings from words.txt
Found 10 classes in val split
Loaded 34 samples from 10 classes
Computing CLIP text embeddings...


Computing embeddings: 100%|██████████| 10/10 [00:00<00:00, 227.35it/s]

Created data loaders:
  Train: 158 batches (316 samples)
  Val: 17 batches (34 samples)

✅ Dataset loading successful!
Available classes (first 10): ['n01440764', 'n01443537', 'n01484850', 'n01491361', 'n01494475', 'n01496331', 'n01498041', 'n01514668', 'n01514859', 'n01518878']

Testing batch loading...
  Batch 1:
    Images shape: torch.Size([2, 3, 256, 256])
    Text embeddings shape: torch.Size([2, 512])
    Class names: ['hammerhead', 'hammerhead']
  Batch 2:
    Images shape: torch.Size([2, 3, 256, 256])
    Text embeddings shape: torch.Size([2, 512])
    Class names: ['hen', 'hammerhead']
  Batch 3:
    Images shape: torch.Size([2, 3, 256, 256])
    Text embeddings shape: torch.Size([2, 512])
    Class names: ['goldfish', 'great white shark']

✅ Dataset test completed successfully!


True

In [5]:
# Enhanced training with comprehensive visualization and optimized performance

# Utility: convert a torch tensor to a PIL Image.
# Accepts tensors in shape (C,H,W), (H,W) or (B,C,H,W) (first image taken).
# Supports tensors in range [-1,1] or [0,1].
from PIL import Image

def tensor_to_pil(tensor):
    """Convert a torch tensor to a PIL Image.
    - If tensor has batch dimension, uses the first image.
    - Handles tensors in [-1,1] or [0,1].
    - Accepts (C,H,W) or (H,W) for single-channel images.
    """
    if not isinstance(tensor, torch.Tensor):
        raise TypeError("tensor_to_pil expects a torch.Tensor")
    t = tensor.detach().cpu()
    if t.ndim == 4:  # B,C,H,W -> take first
        t = t[0]
    if t.ndim == 2:  # H,W -> single channel
        arr = (t * 255.0).clamp(0, 255).to(torch.uint8).numpy()
        return Image.fromarray(arr, mode='L')
    # Now expect C,H,W
    if t.shape[0] == 1:
        t = t.squeeze(0)
        arr = (t * 255.0).clamp(0, 255).to(torch.uint8).numpy()
        return Image.fromarray(arr, mode='L')
    # 3xHxW -> HxWx3
    if t.shape[0] == 3:
        # If values likely in [-1,1], rescale to [0,1]
        if t.min() < 0:
            t = (t + 1.0) / 2.0
        t = t.permute(1, 2, 0)  # H,W,C
        arr = (t * 255.0).clamp(0, 255).to(torch.uint8).numpy()
        return Image.fromarray(arr)
    # Fallback: try to convert whatever shape remains
    t = t.permute(1, 2, 0)
    if t.min() < 0:
        t = (t + 1.0) / 2.0
    arr = (t * 255.0).clamp(0, 255).to(torch.uint8).numpy()
    return Image.fromarray(arr)

def main_train():
    """Main training function with enhanced visualization and optimized performance"""
    
    # OPTIMIZED Configuration for faster training
    config = {
        'batch_size': 8,  # ⚡ INCREASED from 4 to 8 for better GPU utilization
        'learning_rate': 2e-4,  # ⚡ INCREASED learning rate for faster convergence
        'num_epochs': 30,  # ⚡ REDUCED epochs for faster initial training
        'image_size': 256,
        'save_dir': Path('./checkpoints'),
        'imagenet_root': r'D:\DDPM-diffusion\data\imagenet-mini',
        'max_classes': 50,  # ⚡ REDUCED from 100 to 50 for much faster dataset loading
        'num_workers': 0,  # ⚡ SET to 0 to avoid Windows multiprocessing issues
        'gradient_clip': 1.0,
        'save_every': 3,  # ⚡ REDUCED checkpoint frequency
        'demo_every': 5,   # ⚡ REDUCED demo frequency
        'guidance_scale': 7.5,
        # Performance optimizations
        'pin_memory': True,
        'persistent_workers': False,  # Disable for num_workers=0
        'prefetch_factor': 2,
    }
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Using device: {device}")
    print(f"⚡ OPTIMIZED CONFIG: batch_size={config['batch_size']}, max_classes={config['max_classes']}")
    
    # Load CLIP model
    print("📎 Loading CLIP model...")
    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    
    # Create data loaders for ImageNet Mini with optimized settings
    print("📂 Creating OPTIMIZED data loaders...")
    train_loader, val_loader, available_classes = create_imagenet_mini_loaders(
        root_dir=config['imagenet_root'],
        clip_model=clip_model,
        batch_size=config['batch_size'],  # Larger batch
        image_size=config['image_size'],
        num_workers=config['num_workers'],  # No multiprocessing 
        max_classes=config['max_classes']  # Fewer classes for speed
    )
    
    print(f"📊 FAST Dataset loaded with {len(available_classes)} classes")
    print(f"   Training batches: {len(train_loader)} (batch_size={config['batch_size']})")
    print(f"   Validation batches: {len(val_loader)}")
    print(f"   ⚡ Total training samples: {len(train_loader) * config['batch_size']}")
    
    # Create model
    print("🏗️ Creating TextConditionalUNet model...")
    model = TextConditionalUNet(
        text_embed_dim=512,
        model_channels=128,  # Keep model size manageable
        image_size=config['image_size'],
        in_channels=3
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Model parameters: {total_params:,}")
    
    # Create DDIM scheduler
    print("⏰ Initializing DDIM scheduler...")
    scheduler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="linear")
    
    # Setup save directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir = config['save_dir'] / f"fast_training_{timestamp}"
    save_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"💾 Checkpoints will be saved to: {save_dir}")
    
    # Save training configuration
    config_path = save_dir / "training_config.json"
    with open(config_path, 'w') as f:
        config_serializable = {k: str(v) if isinstance(v, Path) else v for k, v in config.items()}
        config_serializable['device'] = str(device)
        config_serializable['model_params'] = total_params
        config_serializable['available_classes'] = len(available_classes)
        json.dump(config_serializable, f, indent=2)
    
    print(f"⚙️ Configuration saved to: {config_path}")
    
    # Train the model with enhanced visualization
    print("\n🎯 Starting OPTIMIZED training with visualization...")
    print(f"   ⚡ Faster settings: {config['num_epochs']} epochs, {config['max_classes']} classes")
    
    final_model_path = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        scheduler=scheduler,
        num_epochs=config['num_epochs'],
        device=device,
        save_dir=save_dir,
        learning_rate=config['learning_rate'],
        clip_model=clip_model  # Pass CLIP model for visualization
    )
    
    print(f"\n✅ FAST Training completed successfully!")
    print(f"📁 All outputs saved to: {save_dir}")
    print(f"🏆 Final model: {final_model_path}")
    
    # Create a quick demo after training
    print("\n🎨 Generating post-training demo samples...")
    try:
        demo_prompts = [
            "a photo of a beautiful golden retriever",
            "a photo of a cute tabby cat",
            "a photo of a red sports car",
            "a photo of a colorful parrot",
            "a photo of a majestic mountain landscape"
        ]
        
        demo_dir = save_dir / "post_training_demo"
        demo_dir.mkdir(exist_ok=True)
        
        # Load the best model for demo
        best_model_path = save_dir / "best_model.pth"
        if best_model_path.exists():
            print(f"🔄 Loading best model for demo: {best_model_path}")
            
            # Create demo model
            demo_model = TextConditionalUNet(
                text_embed_dim=512,
                model_channels=128,
                image_size=256,
                in_channels=3
            ).to(device)
            
            checkpoint = torch.load(best_model_path, map_location=device)
            demo_model.load_state_dict(checkpoint['model_state_dict'])
            demo_model.eval()
            
            # Generate demo samples
            with torch.no_grad():
                for i, prompt in enumerate(demo_prompts):
                    text_tokens = clip.tokenize([prompt]).to(device)
                    text_embedding = clip_model.encode_text(text_tokens)
                    text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
                    
                    generated_img = ddim_sample(
                        model=demo_model,
                        scheduler=scheduler,
                        text_embeddings=text_embedding,
                        num_inference_steps=50,  # High quality
                        cfg_scale=7.5,
                        eta=0.0,
                        device=device
                    )[0]
                    
                    img_pil = tensor_to_pil(generated_img)
                    safe_prompt = "".join(c for c in prompt if c.isalnum() or c in (' ', '-', '_')).strip()
                    safe_prompt = safe_prompt.replace(' ', '_')
                    
                    img_path = demo_dir / f"demo_{i:02d}_{safe_prompt}.png"
                    img_pil.save(img_path)
                    
                    print(f"   ✅ Generated: {prompt}")
            
            print(f"🎨 Demo samples saved to: {demo_dir}")
        
    except Exception as e:
        print(f"⚠️ Post-training demo failed: {e}")
    
    return final_model_path

# Quick dataset test with optimized settings
def test_dataset_fast():
    """Fast dataset test with optimized configuration"""
    print("🧪 Testing ImageNet Mini dataset loading with OPTIMIZED settings...")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    
    dataset_path = r'D:\DDPM-diffusion\data\imagenet-mini'
    
    try:
        train_loader, val_loader, available_classes = create_imagenet_mini_loaders(
            root_dir=dataset_path,
            clip_model=clip_model,
            batch_size=8,  # ⚡ Larger batch
            image_size=256,
            num_workers=0,  # ⚡ No multiprocessing 
            max_classes=20  # ⚡ Even fewer for testing
        )
        
        print(f"\n✅ FAST Dataset loading successful!")
        print(f"   Available classes: {len(available_classes)}")
        print(f"   Training batches: {len(train_loader)} (batch_size=8)")
        print(f"   Validation batches: {len(val_loader)}")
        
        # Test loading batches
        print("\n🚀 Testing fast batch loading...")
        for batch_idx, (images, text_embeddings, class_names) in enumerate(train_loader):
            print(f"   Batch {batch_idx + 1}: {images.shape}, embeddings: {text_embeddings.shape}")
            if batch_idx >= 1:  # Test fewer batches
                break
        
        print("\n✅ FAST Dataset test completed successfully!")
        return True
        
    except Exception as e:
        print(f"\n❌ Dataset test failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run optimized training
print("🚀 OPTIMIZED TRAINING CONFIGURATION:")
print("   - Batch size: 8 (increased for better GPU utilization)")
print("   - Max classes: 50 (reduced for faster dataset loading)")
print("   - Epochs: 30 (reduced for faster initial training)")
print("   - Learning rate: 2e-4 (increased for faster convergence)")
print("   - Workers: 0 (no multiprocessing for stability)")
print("\nStarting training...")

main_train()

🚀 OPTIMIZED TRAINING CONFIGURATION:
   - Batch size: 8 (increased for better GPU utilization)
   - Max classes: 50 (reduced for faster dataset loading)
   - Epochs: 30 (reduced for faster initial training)
   - Learning rate: 2e-4 (increased for faster convergence)
   - Workers: 0 (no multiprocessing for stability)

Starting training...
🚀 Using device: cuda
⚡ OPTIMIZED CONFIG: batch_size=8, max_classes=50
📎 Loading CLIP model...
📂 Creating OPTIMIZED data loaders...
Creating ImageNet Mini datasets from D:\DDPM-diffusion\data\imagenet-mini
Loaded 82115 class mappings from words.txt
Found 50 classes in train split
Loaded 1561 samples from 50 classes
Computing CLIP text embeddings...


Computing embeddings: 100%|██████████| 50/50 [00:00<00:00, 233.07it/s]


Loaded 82115 class mappings from words.txt
Found 50 classes in val split
Loaded 169 samples from 50 classes
Computing CLIP text embeddings...


Computing embeddings: 100%|██████████| 50/50 [00:00<00:00, 189.43it/s]


Created data loaders:
  Train: 195 batches (1561 samples)
  Val: 22 batches (169 samples)
📊 FAST Dataset loaded with 50 classes
   Training batches: 195 (batch_size=8)
   Validation batches: 22
   ⚡ Total training samples: 1560
🏗️ Creating TextConditionalUNet model...
   Model parameters: 217,802,499
⏰ Initializing DDIM scheduler...
💾 Checkpoints will be saved to: checkpoints\fast_training_20250830_145255
⚙️ Configuration saved to: checkpoints\fast_training_20250830_145255\training_config.json

🎯 Starting OPTIMIZED training with visualization...
   ⚡ Faster settings: 30 epochs, 50 classes
📊 TensorBoard logging enabled
🚀 Starting enhanced training on cuda
📊 Model parameters: 217,802,499
🎨 Will generate samples every 1000 steps
💾 Will save checkpoints every 5 epochs


Epoch 1/30:   0%|          | 0/195 [00:25<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 22.11 GiB is allocated by PyTorch, and 201.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)